# 111 — OpenAI Fine-Tuning API
## What you'll learn: the full loop from JSONL prep to base vs. fine-tuned comparison
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/111-openai-finetuning/openai_finetuning_workbook.ipynb)

Fine-tuning lets you train a model on your own labeled examples so it behaves exactly the way you need — without stuffing instructions into every prompt. This workshop walks the full pipeline end to end: synthesizing training data, writing JSONL, uploading it to the OpenAI Files API, launching a fine-tuning job, and comparing the fine-tuned model's accuracy against the base model on a held-out eval set.

**Task**: customer service tone classification — label every incoming message as `formal`, `informal`, or `urgent`.

**Cost**: ~$0.02–$0.05 for a 50-example run on `gpt-4o-mini-2024-07-18`. A `dry_run=True` mode is available to demo the full pipeline without launching an actual training job.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — fine-tuning vs. prompting: when to use which |
| 2 | **Setup** — install deps, configure API key |
| 3 | **Dataset preparation** — JSONL format, synthetic data generation |
| 4 | **Inspecting training data** — validation and format checks |
| 5 | **Upload to Files API** — `client.files.create()` |
| 6 | **Launch fine-tuning job** — dry_run explained |
| 7 | **Poll job status** — monitoring until `succeeded` |
| 8 | **Evaluation** — base vs. fine-tuned accuracy on held-out examples |
| 9 | **Interpreting results** — what the numbers mean |
| 10 | **Decision framework** — when fine-tuning is worth the cost |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `openai`, `python-dotenv`

### Key References
> OpenAI Fine-Tuning Guide — [platform.openai.com/docs/guides/fine-tuning](https://platform.openai.com/docs/guides/fine-tuning)
>
> OpenAI Files API — [platform.openai.com/docs/api-reference/files](https://platform.openai.com/docs/api-reference/files)
>
> Fine-Tuning pricing — [openai.com/pricing](https://openai.com/pricing) (see "Fine-tuning models")

## Part 1 — Concepts: Fine-Tuning vs. Prompting

### What is fine-tuning?

Fine-tuning takes a pre-trained model (like `gpt-4o-mini`) and continues training it on your own labeled examples. The model's weights are updated so it internalises the task — reducing or eliminating the need for detailed in-prompt instructions.

```
Standard workflow (prompt engineering):

  System prompt (200 tokens) → model → output
       ↑
  "You are a tone classifier. Labels are: formal, informal, urgent.
   Here are 3 examples: ..."

  Repeated on every call. Pay for system prompt tokens every time.

Fine-tuned workflow:

  User message (0–10 tokens) → fine-tuned model → output
       ↑
  Task is baked into weights. No system prompt needed.

  Shorter prompts. Faster inference. Consistent format.
```

### When to use fine-tuning

| Scenario | Use fine-tuning | Use prompting |
|----------|-----------------|---------------|
| Task is narrow and well-defined | ✓ | |
| You have 50+ labeled examples | ✓ | |
| Same system prompt repeated 100k+ times/day | ✓ | |
| Task requires output format consistency | ✓ | |
| Task changes week to week | | ✓ |
| You have < 20 examples | | ✓ |
| Exploration / prototyping phase | | ✓ |
| Zero-shot capability is sufficient | | ✓ |

### Fine-tuning pipeline overview

```
┌─────────────────────────────────────────────────────────────┐
│  OpenAI Fine-Tuning Pipeline                                │
│                                                             │
│  1. Prepare data → list of {messages: [...]} dicts          │
│  2. Write JSONL → one JSON object per line                  │
│  3. Upload → client.files.create(purpose="fine-tune")       │
│  4. Launch job → client.fine_tuning.jobs.create(...)        │
│  5. Poll → job.status in ("succeeded", "failed")            │
│  6. Deploy → use fine_tuned_model name like any model       │
│  7. Evaluate → compare accuracy vs. base model              │
└─────────────────────────────────────────────────────────────┘
```

### The JSONL format (required)

OpenAI's fine-tuning API requires **JSONL** (JSON Lines) — one complete JSON object per line. Each object must have a `messages` key containing a list of role/content pairs in chat format:

```json
{"messages": [{"role": "system", "content": "You are a tone classifier..."}, {"role": "user", "content": "Where is my order?"}, {"role": "assistant", "content": "informal"}]}
{"messages": [{"role": "system", "content": "You are a tone classifier..."}, {"role": "user", "content": "URGENT: fix this NOW"}, {"role": "assistant", "content": "urgent"}]}
```

**Key constraints**:
- Minimum 10 examples (50+ recommended for reliable results)
- `assistant` turn must be the final message
- File size limit: 1 GB
- Token limit per example: model's context window

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from openai import OpenAI

client = OpenAI()

# Verify connectivity with a lightweight models list call
models = client.models.list()
print(f"Connected — {len(list(models))} models available.")
print("Ready to fine-tune.")

## Part 3 — Dataset Preparation

### Task: customer service tone classification

We classify each incoming customer service message into one of three labels:

| Label | Meaning | Example |
|-------|---------|---------|
| `formal` | Professional, polite, structured | "I am writing to inquire about order #12345." |
| `informal` | Casual, conversational | "Hey where's my package lol" |
| `urgent` | Time-critical, emphatic, distressed | "EMERGENCY: fix this NOW or I dispute the charge" |

### Generating synthetic training examples

The `generate_training_examples()` function in `src/tools.py` creates balanced synthetic data by:
1. Cycling through the three labels in order (ensuring balance)
2. Sampling from per-label template banks (10 templates each)
3. Filling in dynamic values (`order_id`, `date`) via `str.format()`

Each output example is a dict with a `messages` key — exactly the shape OpenAI's fine-tuning API requires.

```python
# Structure of one training example:
{
    "messages": [
        {"role": "system",    "content": "You are a tone classifier..."},
        {"role": "user",      "content": "I am writing to inquire about order #45231."},
        {"role": "assistant", "content": "formal"},
    ]
}
```

In [ ]:
# ===== src/tools.py — generate_training_examples =====

import json
import random
import time
from pathlib import Path

LABELS = ["formal", "informal", "urgent"]

FORMAL_TEMPLATES = [
    "I am writing to inquire about the status of my order #{order_id}.",
    "Could you please provide an update regarding my recent purchase?",
    "I would like to request information about your return policy.",
    "Please advise on the estimated delivery timeline for order #{order_id}.",
    "I wish to formally request a refund for the defective item received.",
    "Kindly confirm receipt of my previous correspondence.",
    "I am following up on the unresolved issue submitted on {date}.",
    "May I inquire about the terms and conditions for warranty claims?",
    "I respectfully request escalation of this matter to a senior representative.",
    "Please provide written confirmation of the agreed resolution.",
]

INFORMAL_TEMPLATES = [
    "Hey, where's my order? It's been a week!",
    "Just checking in — did you guys get my return?",
    "Hi! Can you help me figure out what happened to my package?",
    "Any update on when my stuff will show up?",
    "Hey there, I need help changing my delivery address.",
    "Quick question — can I swap this for a different color?",
    "Yo, my order shows delivered but I don't have it??",
    "Hey can someone look into this for me please?",
    "Got the wrong item lol, how do I fix this?",
    "Just wondering if there's a discount code I can use.",
]

URGENT_TEMPLATES = [
    "URGENT: My event is tomorrow and the order hasn't arrived!!!",
    "This is critical — I need this resolved TODAY or I'm disputing the charge.",
    "EMERGENCY: The item I received is dangerous and I need an immediate callback.",
    "I have been waiting 3 weeks. This needs to be fixed RIGHT NOW.",
    "My client presentation is in 2 hours and the product is missing. Help!!",
    "Critical issue: the system is sending me duplicate charges, need immediate fix.",
    "I need a supervisor NOW — this has gone on long enough.",
    "ASAP — I'm at the store trying to return this and your system is down.",
    "My order is perishable and it's now 4 days late. I need action immediately.",
    "CRITICAL: Wrong medication dosage label — need callback within the hour.",
]


def generate_training_examples(n: int = 50, seed: int = 42) -> list[dict]:
    """
    Generate N synthetic training examples for tone classification.
    Each example: {messages: [{role, content}, {role, content}]}
    Labels are balanced across formal / informal / urgent.
    """
    random.seed(seed)
    examples = []
    templates = {
        "formal": FORMAL_TEMPLATES,
        "informal": INFORMAL_TEMPLATES,
        "urgent": URGENT_TEMPLATES,
    }
    system_msg = (
        "You are a tone classifier for customer service messages. "
        "Classify each message as exactly one of: formal, informal, urgent. "
        "Reply with only the label — no explanation."
    )
    for i in range(n):
        label = LABELS[i % len(LABELS)]
        tmpl = random.choice(templates[label])
        message = tmpl.format(order_id=random.randint(10000, 99999), date="2024-01-15")
        examples.append({
            "messages": [
                {"role": "system",    "content": system_msg},
                {"role": "user",      "content": message},
                {"role": "assistant", "content": label},
            ]
        })
    return examples


# Generate and inspect
train_examples = generate_training_examples(n=50)
print(f"Generated {len(train_examples)} training examples")
print(f"\nExample 0 (formal):")
for msg in train_examples[0]["messages"]:
    print(f"  [{msg['role']:9}] {msg['content'][:80]}")
print(f"\nExample 1 (informal):")
for msg in train_examples[1]["messages"]:
    print(f"  [{msg['role']:9}] {msg['content'][:80]}")
print(f"\nExample 2 (urgent):")
for msg in train_examples[2]["messages"]:
    print(f"  [{msg['role']:9}] {msg['content'][:80]}")

## Part 4 — Inspecting Training Data

Before uploading, it's worth validating the data: label distribution, message length distribution, and JSONL format. OpenAI rejects uploads with malformed examples — catching issues locally saves a round-trip.

**Validation checklist**:
- Every example has a `messages` key
- Every message has `role` and `content`
- `assistant` is always the last message
- No empty content strings
- Labels are exactly one of the expected values

In [ ]:
# Validate and inspect the training set

from collections import Counter

def validate_training_examples(examples: list[dict]) -> dict:
    """Validate structure and label balance of training examples."""
    errors = []
    label_counts = Counter()

    for i, ex in enumerate(examples):
        if "messages" not in ex:
            errors.append(f"Example {i}: missing 'messages' key")
            continue
        msgs = ex["messages"]
        if not msgs:
            errors.append(f"Example {i}: empty messages list")
            continue
        if msgs[-1]["role"] != "assistant":
            errors.append(f"Example {i}: last role is '{msgs[-1]['role']}', expected 'assistant'")
        for msg in msgs:
            if not msg.get("content", "").strip():
                errors.append(f"Example {i}: empty content in role '{msg['role']}'")
        label = msgs[-1]["content"].strip().lower()
        label_counts[label] += 1
        if label not in LABELS:
            errors.append(f"Example {i}: unexpected label '{label}'")

    return {"errors": errors, "label_counts": dict(label_counts), "total": len(examples)}


validation = validate_training_examples(train_examples)
print(f"Total examples: {validation['total']}")
print(f"Errors: {len(validation['errors'])}")
if validation["errors"]:
    for e in validation["errors"][:5]:
        print(f"  {e}")
else:
    print("  None — all examples are valid")
print(f"\nLabel distribution:")
for label, count in sorted(validation["label_counts"].items()):
    bar = "█" * count
    print(f"  {label:10} {count:3}  {bar}")

In [ ]:
# Inspect message length distribution (proxy for token cost)

msg_lengths = [len(ex["messages"][1]["content"]) for ex in train_examples]
print(f"User message lengths (chars):")
print(f"  Min:    {min(msg_lengths)}")
print(f"  Max:    {max(msg_lengths)}")
print(f"  Mean:   {sum(msg_lengths) / len(msg_lengths):.1f}")
print(f"  Median: {sorted(msg_lengths)[len(msg_lengths)//2]}")

# Rough token estimate: ~4 chars per token for English
# System prompt + user + assistant label ≈ 60 + avg_user + 2 tokens per example
system_tokens = len(train_examples[0]["messages"][0]["content"]) // 4
avg_user_tokens = int(sum(msg_lengths) / len(msg_lengths)) // 4
avg_tokens_per_example = system_tokens + avg_user_tokens + 2
total_training_tokens = avg_tokens_per_example * len(train_examples)

print(f"\nToken estimates (rough: 4 chars ≈ 1 token):")
print(f"  System prompt:        ~{system_tokens} tokens")
print(f"  Avg user message:     ~{avg_user_tokens} tokens")
print(f"  Avg per example:      ~{avg_tokens_per_example} tokens")
print(f"  Total training set:   ~{total_training_tokens} tokens")
print(f"\nEstimated fine-tuning cost at $0.003/1K training tokens × 3 epochs:")
cost = (total_training_tokens * 3 / 1000) * 0.003
print(f"  ~${cost:.4f}")

## Part 5 — Writing JSONL and Uploading to the Files API

### Step 1: Write JSONL

`write_jsonl()` serializes each example dict to a single JSON line. This is the format the Files API expects — one complete JSON object per line, no commas between objects, no outer array.

### Step 2: Upload

`upload_training_file()` calls `client.files.create(file=..., purpose="fine-tune")` and returns the `file_id` (format: `file-abc123...`). The file stays on OpenAI's servers — you reference it by ID when creating the fine-tuning job.

```
Files API response:
  {
    "id": "file-abc123xyz",
    "object": "file",
    "purpose": "fine-tune",
    "status": "processed",
    "bytes": 12345,
    "created_at": 1715000000
  }
```

In [ ]:
def write_jsonl(examples: list[dict], path: Path) -> Path:
    """Write training examples to JSONL format required by OpenAI."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    return path


def upload_training_file(client: OpenAI, jsonl_path: Path) -> str:
    """Upload JSONL to OpenAI Files API. Returns file_id."""
    with open(jsonl_path, "rb") as f:
        response = client.files.create(file=f, purpose="fine-tune")
    return response.id


# Write JSONL
import tempfile
data_dir = Path(tempfile.mkdtemp())
jsonl_path = data_dir / "train.jsonl"
write_jsonl(train_examples, jsonl_path)

# Verify the file
lines = jsonl_path.read_text().strip().split("\n")
print(f"JSONL written: {jsonl_path}")
print(f"Lines: {len(lines)}")
print(f"File size: {jsonl_path.stat().st_size:,} bytes")
print(f"\nFirst line preview:")
first = json.loads(lines[0])
print(f"  Keys: {list(first.keys())}")
print(f"  Roles: {[m['role'] for m in first['messages']]}")
print(f"  Label: {first['messages'][-1]['content']}")

In [ ]:
# Upload to OpenAI Files API

print("Uploading to OpenAI Files API...")
file_id = upload_training_file(client, jsonl_path)
print(f"Upload complete.")
print(f"File ID: {file_id}")

# Retrieve file metadata to confirm
file_meta = client.files.retrieve(file_id)
print(f"Status:  {file_meta.status}")
print(f"Bytes:   {file_meta.bytes:,}")
print(f"Purpose: {file_meta.purpose}")

## Part 6 — Launching a Fine-Tuning Job

### The dry_run pattern

Fine-tuning takes 5–20 minutes and costs real money. In demos and CI environments, `dry_run=True` lets you exercise the full data pipeline (generate → write → upload) without launching an actual training job.

```
dry_run=True   →  stops after upload, returns file_id immediately (~$0.00)
dry_run=False  →  launches training job, polls until done (~$0.02-0.05 for 50 examples)
```

### What `create_finetuning_job()` does

```python
job = client.fine_tuning.jobs.create(
    training_file=file_id,           # from Files API upload
    model="gpt-4o-mini-2024-07-18",  # base model to fine-tune
    hyperparameters={"n_epochs": 3}, # default: auto; 3 is usually sufficient
)
# job.id → "ftjob-abc123..."
```

**Hyperparameters**:
- `n_epochs`: number of full passes over the training set. 3–5 is typical for small datasets.
- `learning_rate_multiplier`: defaults to `auto` (OpenAI selects based on dataset size)
- `batch_size`: defaults to `auto`

### Job lifecycle

```
queued → running → succeeded
                 ↘ failed
                 ↘ cancelled
```

Jobs typically queue immediately and start training within seconds to minutes depending on load.

In [ ]:
def create_finetuning_job(client: OpenAI, file_id: str, model: str = "gpt-4o-mini-2024-07-18") -> str:
    """Launch a fine-tuning job. Returns job_id."""
    job = client.fine_tuning.jobs.create(
        training_file=file_id,
        model=model,
        hyperparameters={"n_epochs": 3},
    )
    return job.id


# ── DRY RUN MODE ──
# Set dry_run = False to actually launch training (costs ~$0.02-0.05)
DRY_RUN = True
BASE_MODEL = "gpt-4o-mini-2024-07-18"

if DRY_RUN:
    print("[DRY RUN] Stopping before job creation.")
    print(f"  File ID ready for training: {file_id}")
    print(f"  Base model:                 {BASE_MODEL}")
    print(f"  To train: set DRY_RUN = False in this cell")
    job_id = None
else:
    print(f"Launching fine-tuning job on {BASE_MODEL}...")
    job_id = create_finetuning_job(client, file_id, model=BASE_MODEL)
    print(f"Job created: {job_id}")
    print(f"Monitor at: https://platform.openai.com/finetune")

## Part 7 — Polling Job Status

Fine-tuning is asynchronous. After creating a job, you poll `client.fine_tuning.jobs.retrieve(job_id)` until `status` is `succeeded` or `failed`.

### What `poll_job_until_done()` does

```
┌─────────────────────────────────────────────────────────┐
│  poll_job_until_done(client, job_id, poll_interval=30)  │
│                                                         │
│  loop:                                                  │
│    job = client.fine_tuning.jobs.retrieve(job_id)       │
│    print(job.status)                                    │
│    if status in ("succeeded", "failed", "cancelled"):   │
│      return {status, fine_tuned_model, trained_tokens}  │
│    sleep(30)  # poll every 30 seconds                   │
└─────────────────────────────────────────────────────────┘
```

### What the result looks like on success

```python
{
    "status": "succeeded",
    "fine_tuned_model": "ft:gpt-4o-mini-2024-07-18:org::abc123xyz",
    "trained_tokens": 4512,
    "error": None,
}
```

The `fine_tuned_model` string is your new model name — use it exactly like any other OpenAI model identifier in subsequent API calls.

### Streaming events (alternative to polling)

```python
# You can also stream training events in real time:
for event in client.fine_tuning.jobs.list_events(job_id=job_id, limit=50):
    print(event.message)
# Events include: "Step X/Y: training loss=0.42", "New checkpoints created", etc.
```

In [ ]:
def poll_job_until_done(client: OpenAI, job_id: str, poll_interval: int = 30) -> dict:
    """Poll fine-tuning job until status is succeeded or failed."""
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        print(f"  Job {job_id[:20]}... status: {status}")
        if status in ("succeeded", "failed", "cancelled"):
            return {
                "status": status,
                "fine_tuned_model": getattr(job, "fine_tuned_model", None),
                "trained_tokens": getattr(job, "trained_tokens", None),
                "error": getattr(job, "error", None),
            }
        time.sleep(poll_interval)


# Demonstrate polling against a real job (only runs when DRY_RUN = False)
if job_id is not None:
    print(f"Polling job {job_id}...")
    print("(This typically takes 5-20 minutes — training loss updates appear as events)")
    job_result = poll_job_until_done(client, job_id)
    print(f"\nJob complete:")
    print(f"  Status:          {job_result['status']}")
    print(f"  Fine-tuned model: {job_result['fine_tuned_model']}")
    print(f"  Trained tokens:  {job_result['trained_tokens']}")
    FINETUNED_MODEL = job_result["fine_tuned_model"]
else:
    # DRY RUN: simulate what a completed result looks like
    FINETUNED_MODEL = None
    print("[DRY RUN] Simulating a completed job result:")
    simulated = {
        "status": "succeeded",
        "fine_tuned_model": "ft:gpt-4o-mini-2024-07-18:my-org::simulated123",
        "trained_tokens": 4512,
        "error": None,
    }
    for k, v in simulated.items():
        print(f"  {k}: {v}")

## Part 8 — Generating Eval Examples and Running Evaluation

### Held-out eval set

`generate_eval_examples()` creates 12 examples that are **not** in the training set — different templates, same label space. The eval format is simpler: `{message, label}` dicts (no `messages` wrapping, since we classify them live).

### `classify_message()` — single inference call

Each model is called with the same system prompt and the user message, temperature=0 for reproducibility, and `max_tokens=10` (the label is only 1 token).

### `run_evaluation()` — head-to-head comparison

```
for each eval example:
    base_pred    = classify_message(client, base_model, message)
    finetuned_pred = classify_message(client, finetuned_model, message)
    compare both to ground truth label

return:
    base_accuracy, finetuned_accuracy, improvement
    per-example breakdown for both models
```

### Why exact match scoring is appropriate here

Our labels (`formal`, `informal`, `urgent`) are closed-vocabulary — any output not matching exactly is wrong. We don't need semantic similarity metrics.

In [ ]:
def generate_eval_examples(n: int = 12, seed: int = 99) -> list[dict]:
    """Generate N held-out evaluation examples (not in training set)."""
    random.seed(seed)
    examples = []
    templates = {
        "formal": [
            "I would appreciate your earliest response to my outstanding query.",
            "Please provide documentation confirming my subscription cancellation.",
            "I am requesting a formal acknowledgment of the delay in my order.",
            "Kindly review the attached invoice discrepancy at your earliest convenience.",
        ],
        "informal": [
            "Can u guys check on my order? Thanks!",
            "Lol I think I ordered the wrong size, can I change it?",
            "Hey is there any way to track my package?",
            "Oh no I think I gave you the wrong address — help!",
        ],
        "urgent": [
            "I NEED THIS FIXED IMMEDIATELY — my business depends on it.",
            "Emergency: the device exploded and I need a recall report NOW.",
            "DROP EVERYTHING — I've been double-billed $500 and need it reversed today.",
            "URGENT URGENT URGENT: wrong address, delivery attempt tomorrow morning!!!",
        ],
    }
    for i in range(n):
        label = LABELS[i % len(LABELS)]
        msg = random.choice(templates[label])
        examples.append({"message": msg, "label": label})
    return examples


eval_examples = generate_eval_examples(n=12)
print(f"Generated {len(eval_examples)} eval examples")
print(f"\nEval set:")
print(f"{'#':>3}  {'Label':10}  {'Message'}")
print("-" * 75)
for i, ex in enumerate(eval_examples):
    print(f"{i+1:>3}  {ex['label']:10}  {ex['message'][:55]}")

In [ ]:
def classify_message(client: OpenAI, model: str, message: str) -> str:
    """Run single tone classification call. Returns predicted label."""
    system_msg = (
        "You are a tone classifier for customer service messages. "
        "Classify each message as exactly one of: formal, informal, urgent. "
        "Reply with only the label — no explanation."
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": message},
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip().lower()


def run_evaluation(
    client: OpenAI, base_model: str, finetuned_model: str, eval_examples: list[dict]
) -> dict:
    """
    Run the same eval set against both models.
    Returns accuracy scores and per-example breakdown.
    """
    base_results = []
    ft_results = []
    for ex in eval_examples:
        msg = ex["message"]
        expected = ex["label"]
        base_pred = classify_message(client, base_model, msg)
        ft_pred = classify_message(client, finetuned_model, msg)
        base_results.append({
            "message":   msg[:60],
            "expected":  expected,
            "predicted": base_pred,
            "correct":   base_pred == expected,
        })
        ft_results.append({
            "message":   msg[:60],
            "expected":  expected,
            "predicted": ft_pred,
            "correct":   ft_pred == expected,
        })
    base_acc = sum(r["correct"] for r in base_results) / len(base_results)
    ft_acc   = sum(r["correct"] for r in ft_results)   / len(ft_results)
    return {
        "base_model":       base_model,
        "finetuned_model":  finetuned_model,
        "n_eval":           len(eval_examples),
        "base_accuracy":    base_acc,
        "finetuned_accuracy": ft_acc,
        "improvement":      ft_acc - base_acc,
        "base_results":     base_results,
        "ft_results":       ft_results,
    }


# Only run full evaluation when a real fine-tuned model is available
if FINETUNED_MODEL is not None:
    print(f"Running evaluation: {BASE_MODEL} vs {FINETUNED_MODEL}")
    print(f"Eval set size: {len(eval_examples)} examples ({len(eval_examples)*2} API calls)")
    eval_results = run_evaluation(client, BASE_MODEL, FINETUNED_MODEL, eval_examples)
else:
    # DRY RUN: run base model only, simulate fine-tuned predictions
    print("[DRY RUN] Running base model evaluation only...")
    print(f"(Set DRY_RUN=False and complete training to get fine-tuned results)\n")
    base_only = []
    for ex in eval_examples:
        pred = classify_message(client, BASE_MODEL, ex["message"])
        correct = pred == ex["label"]
        base_only.append({"message": ex["message"][:60], "expected": ex["label"], "predicted": pred, "correct": correct})
        status = "OK" if correct else "WRONG"
        print(f"  [{status}] expected={ex['label']:8} predicted={pred:8}  {ex['message'][:45]}")
    base_acc = sum(r["correct"] for r in base_only) / len(base_only)
    print(f"\nBase model accuracy: {base_acc:.0%} ({sum(r['correct'] for r in base_only)}/{len(base_only)})")
    eval_results = None

## Part 9 — Interpreting Results

### Reading the comparison table

When `eval_results` is available (after a real training run), the table shows:

```
                              Base model    Fine-tuned
Accuracy                         75%           92%
Improvement                                   +17%
```

**What these numbers mean**:

- **Base model accuracy at 75%**: `gpt-4o-mini` can classify tone zero-shot reasonably well, but makes systematic errors — often confusing polite informal messages with formal ones, or missing urgency cues that don't use UPPERCASE.
- **Fine-tuned accuracy at 92%**: After seeing 50 labeled examples across 3 epochs (~150 gradient steps), the model has internalised the task's specific conventions.
- **17% improvement**: Represents the delta from task-specific supervision.

### Per-example breakdown analysis

Look for **error patterns** in the incorrect predictions:
- Which label is most often confused with which?
- Are errors concentrated on edge cases (e.g., a very polite urgent message)?
- Does the fine-tuned model make the same errors as the base model, or different ones?

A confusion matrix helps here:

```
                 Predicted
Actual       formal  informal  urgent
formal          4        0       0
informal        0        3       1   ← 1 informal classified as urgent
urgent          0        0       4
```

In [ ]:
# Display full results when available

def print_comparison_table(results: dict) -> None:
    """Pretty-print the base vs. fine-tuned comparison."""
    print(f"\n{'='*65}")
    print(f"Base model:      {results['base_model']}")
    print(f"Fine-tuned:      {results['finetuned_model']}")
    print(f"Eval set size:   {results['n_eval']} examples")
    print(f"{'='*65}")
    print(f"  Base accuracy:        {results['base_accuracy']:.0%}  "
          f"({sum(r['correct'] for r in results['base_results'])}/{results['n_eval']})")
    print(f"  Fine-tuned accuracy:  {results['finetuned_accuracy']:.0%}  "
          f"({sum(r['correct'] for r in results['ft_results'])}/{results['n_eval']})")
    print(f"  Improvement:          {results['improvement']:+.0%}")
    print(f"{'='*65}")

    print(f"\n{'Per-example breakdown':}")
    print(f"{'#':>3}  {'Expected':10}  {'Base':10}  {'FT':10}  {'Δ':5}  Message")
    print("-" * 80)
    for i, (b, f) in enumerate(zip(results["base_results"], results["ft_results"])):
        base_ok = "OK  " if b["correct"] else "WRONG"
        ft_ok   = "OK  " if f["correct"] else "WRONG"
        delta   = ("  +1" if f["correct"] and not b["correct"] else
                   "  -1" if not f["correct"] and b["correct"] else
                   "   0")
        print(f"{i+1:>3}  {b['expected']:10}  {b['predicted']:10}  {f['predicted']:10}  {delta}  {b['message'][:35]}")


def build_confusion_matrix(results: list[dict]) -> dict:
    """Build a confusion matrix from per-example results."""
    matrix = {true: {pred: 0 for pred in LABELS} for true in LABELS}
    for r in results:
        expected = r["expected"]
        predicted = r["predicted"]
        if predicted in LABELS:
            matrix[expected][predicted] += 1
    return matrix


def print_confusion_matrix(matrix: dict, title: str) -> None:
    print(f"\n{title}")
    print(f"{'':14}", end="")
    for label in LABELS:
        print(f"{label:12}", end="")
    print()
    for true_label in LABELS:
        print(f"  {true_label:12}", end="")
        for pred_label in LABELS:
            count = matrix[true_label][pred_label]
            marker = f"[{count}]" if true_label == pred_label else f" {count} "
            print(f"{marker:12}", end="")
        print()


if eval_results is not None:
    print_comparison_table(eval_results)
    cm_base = build_confusion_matrix(eval_results["base_results"])
    cm_ft   = build_confusion_matrix(eval_results["ft_results"])
    print_confusion_matrix(cm_base, "Confusion Matrix — Base Model")
    print_confusion_matrix(cm_ft,   "Confusion Matrix — Fine-Tuned")
else:
    print("[DRY RUN] Full comparison table requires a completed fine-tuning job.")
    print("Run with DRY_RUN=False to see the base vs. fine-tuned comparison.")

## Part 10 — When to Use Fine-Tuning: Decision Framework

Not every task benefits from fine-tuning. Use this framework to decide:

### The fine-tuning decision tree

```
Is your task narrow, well-defined, and stable?
  No  → use prompting. Fine-tuning is overkill.
  Yes ↓

Do you have 50+ labeled examples (or can you generate them)?
  No  → collect data first, or use few-shot prompting.
  Yes ↓

Is the task run frequently (100k+ calls/month)?
  No  → prompt engineering cost is manageable; fine-tune only if quality is insufficient.
  Yes ↓ (cost of long system prompts starts to matter)

Does the base model fail on this task even with good prompting?
  No  → prompting may be sufficient. Try before training.
  Yes → fine-tune.
```

### Cost vs. quality trade-off

| Approach | Latency | $/call | Quality | Iteration speed |
|----------|---------|--------|---------|-----------------|
| Zero-shot | Fast | Low | Good | Instant |
| Few-shot (5 ex) | Fast | Medium | Better | Minutes |
| Fine-tuned | Fast | Low | Best | Days (data + train) |
| RAG + prompt | Medium | Medium | Good | Hours |

Fine-tuning has a one-time training cost but **lower per-call cost** at scale (shorter prompts).

### When fine-tuning is NOT the answer

- **Data quality is the bottleneck**: 50 noisy examples produce a noisier model. Label quality matters more than quantity.
- **Task drift**: if the task definition changes monthly, you'll retrain constantly. Prompting is easier to update.
- **Rare edge cases**: fine-tuning improves average-case performance; few-shot examples in the prompt are better for handling specific edge cases.
- **No eval set**: you can't measure improvement without held-out data. Build the eval first, then decide whether fine-tuning is worth it.

In [ ]:
# Full pipeline runner (mirrors create_workflow() from src/workflow.py)

def run_full_pipeline(dry_run: bool = True, base_model: str = "gpt-4o-mini-2024-07-18") -> dict:
    """
    Full OpenAI fine-tuning pipeline.

    Stages:
      1. Generate synthetic JSONL training data (50 examples)
      2. Upload to OpenAI Files API
      3. Launch fine-tuning job (skipped if dry_run=True)
      4. Poll until complete
      5. Run evaluation: base vs. fine-tuned accuracy

    dry_run=True: stops after upload (~$0.00).
    dry_run=False: full training run (~$0.02-0.05 on gpt-4o-mini, 50 examples).
    """
    local_client = OpenAI()
    out_dir = Path(tempfile.mkdtemp())

    print("Stage 1: Generating training data...")
    train_ex = generate_training_examples(n=50)
    eval_ex  = generate_eval_examples(n=12)
    jsonl_p  = out_dir / "train.jsonl"
    write_jsonl(train_ex, jsonl_p)
    print(f"  {len(train_ex)} training + {len(eval_ex)} eval examples | JSONL: {jsonl_p}")

    print("\nStage 2: Uploading to OpenAI Files API...")
    fid = upload_training_file(local_client, jsonl_p)
    print(f"  File ID: {fid}")

    if dry_run:
        print("\n[DRY RUN] Stopping before job creation (dry_run=True).")
        return {"stage": "uploaded", "file_id": fid, "train_count": len(train_ex), "eval_count": len(eval_ex)}

    print("\nStage 3: Launching fine-tuning job...")
    jid = create_finetuning_job(local_client, fid, model=base_model)
    print(f"  Job ID: {jid}")

    print("\nStage 4: Polling (5-20 min typically)...")
    job_res = poll_job_until_done(local_client, jid)
    if job_res["status"] != "succeeded":
        print(f"  Job failed: {job_res}")
        return job_res

    ft_model = job_res["fine_tuned_model"]
    print(f"  Fine-tuned model: {ft_model}")
    print(f"  Trained tokens:   {job_res['trained_tokens']}")

    print("\nStage 5: Evaluating (base vs. fine-tuned)...")
    res = run_evaluation(local_client, base_model, ft_model, eval_ex)

    print(f"\n{'='*50}")
    print(f"Base accuracy:        {res['base_accuracy']:.0%}")
    print(f"Fine-tuned accuracy:  {res['finetuned_accuracy']:.0%}")
    print(f"Improvement:          {res['improvement']:+.0%}")
    print(f"{'='*50}")
    return res


# Run the pipeline (dry_run=True by default — safe to run in Colab)
result = run_full_pipeline(dry_run=True)
print(f"\nPipeline returned: {result}")

## Exercises

### Exercise 1 — Add a new label: `neutral`

The current dataset has 3 labels: `formal`, `informal`, `urgent`. Extend it with a 4th label `neutral` for messages that don't fit the other three (e.g., "Hi, I'd like to check my order status.").

Tasks:
1. Add 5+ `neutral` templates to a `NEUTRAL_TEMPLATES` list
2. Update `LABELS = ["formal", "informal", "urgent", "neutral"]`
3. Update `generate_training_examples()` to include neutral examples (maintaining balance across 4 classes)
4. Verify the label distribution with `validate_training_examples()`

---

### Exercise 2 — Streaming training events

When a real fine-tuning job is running, you can stream its events live instead of polling. Implement `stream_training_events(client, job_id, max_events=20)` that:
1. Calls `client.fine_tuning.jobs.list_events(job_id=job_id, limit=max_events)`
2. Prints each event's timestamp and message
3. Highlights events containing "loss" in a different format (those are the training step updates)

Then explain: what information do training loss events give you, and how would you use them to decide whether to stop training early?

---

### Exercise 3 — Per-label accuracy breakdown

`run_evaluation()` returns overall accuracy but not per-label accuracy. Implement `per_label_accuracy(results: list[dict]) -> dict` that takes a list of per-example result dicts (same format as `eval_results["base_results"]`) and returns:

```python
{
    "formal":   {"correct": 4, "total": 4, "accuracy": 1.0},
    "informal": {"correct": 3, "total": 4, "accuracy": 0.75},
    "urgent":   {"correct": 4, "total": 4, "accuracy": 1.0},
}
```

Then call it on both `base_results` and `ft_results` (when available) and identify which label benefits most from fine-tuning.

In [ ]:
# ===== ANSWER KEY — Exercise 1: Add neutral label =====

NEUTRAL_TEMPLATES = [
    "Hi, I'd like to check on my order status.",
    "Can you tell me when my package is expected to arrive?",
    "I'm looking for information about your refund process.",
    "Could you help me find out if my item is still in stock?",
    "I need to update my shipping address for an upcoming order.",
    "What are your customer service hours?",
    "I'd like to know the dimensions of the product before ordering.",
    "Can you confirm whether my subscription is active?",
]

LABELS_EX1 = ["formal", "informal", "urgent", "neutral"]


def generate_training_examples_v2(n: int = 60, seed: int = 42) -> list[dict]:
    """Extended generator with 4 labels including neutral."""
    random.seed(seed)
    examples = []
    templates = {
        "formal":   FORMAL_TEMPLATES,
        "informal": INFORMAL_TEMPLATES,
        "urgent":   URGENT_TEMPLATES,
        "neutral":  NEUTRAL_TEMPLATES,
    }
    system_msg = (
        "You are a tone classifier for customer service messages. "
        "Classify each message as exactly one of: formal, informal, urgent, neutral. "
        "Reply with only the label — no explanation."
    )
    for i in range(n):
        label = LABELS_EX1[i % len(LABELS_EX1)]
        tmpl = random.choice(templates[label])
        message = tmpl.format(order_id=random.randint(10000, 99999), date="2024-01-15")
        examples.append({
            "messages": [
                {"role": "system",    "content": system_msg},
                {"role": "user",      "content": message},
                {"role": "assistant", "content": label},
            ]
        })
    return examples


# Test the extended generator
ex1_examples = generate_training_examples_v2(n=60)
label_counts = Counter(ex["messages"][-1]["content"] for ex in ex1_examples)
print("Exercise 1 — Extended dataset with 4 labels:")
print(f"Total examples: {len(ex1_examples)}")
print(f"\nLabel distribution:")
for label in LABELS_EX1:
    count = label_counts.get(label, 0)
    bar = "█" * count
    print(f"  {label:10} {count:3}  {bar}")
print("\nSample neutral example:")
neutral_ex = next(ex for ex in ex1_examples if ex["messages"][-1]["content"] == "neutral")
for msg in neutral_ex["messages"]:
    print(f"  [{msg['role']:9}] {msg['content'][:80]}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: Streaming training events =====

def stream_training_events(client: OpenAI, job_id: str, max_events: int = 20) -> None:
    """
    Print training events for a fine-tuning job.
    Events containing 'loss' are training step updates — highlighted differently.
    """
    events = client.fine_tuning.jobs.list_events(job_id=job_id, limit=max_events)
    event_list = list(events)

    if not event_list:
        print("No events found yet. Job may still be queued.")
        return

    print(f"Training events for job {job_id[:20]}... ({len(event_list)} events)")
    print("-" * 60)
    for event in reversed(event_list):  # oldest first
        ts = event.created_at
        msg = event.message
        if "loss" in msg.lower():
            # Training step — highlight with prefix
            print(f"  [STEP] {msg}")
        else:
            print(f"  [INFO] {msg}")

    print("\nHow to use training loss events:")
    print("  - Loss should decrease across steps (convergence)")
    print("  - If loss plateaus early (e.g., after epoch 1), fewer epochs may suffice")
    print("  - If loss never decreases, data quality or label noise is the likely cause")
    print("  - Very low loss (<0.01) may indicate overfitting on small datasets")


# Demonstration (only executes when a real job_id is available)
if job_id is not None:
    stream_training_events(client, job_id, max_events=20)
else:
    print("[DRY RUN] stream_training_events() requires a real job_id.")
    print("Simulating what training events look like:\n")
    simulated_events = [
        "[INFO] Fine-tuning job started",
        "[INFO] Step 1/150: training loss=1.1032",
        "[STEP] Step 25/150: training loss=0.8421",
        "[STEP] Step 50/150: training loss=0.5987",
        "[STEP] Step 75/150: training loss=0.4213",
        "[INFO] New checkpoint created at step 75",
        "[STEP] Step 100/150: training loss=0.3104",
        "[STEP] Step 125/150: training loss=0.2208",
        "[STEP] Step 150/150: training loss=0.1893",
        "[INFO] Fine-tuning job succeeded",
    ]
    for e in simulated_events:
        print(f"  {e}")

In [ ]:
# ===== ANSWER KEY — Exercise 3: Per-label accuracy breakdown =====

def per_label_accuracy(results: list[dict]) -> dict:
    """
    Compute per-label accuracy from a list of per-example result dicts.
    Each dict must have 'expected' and 'correct' keys.

    Returns:
        {label: {"correct": int, "total": int, "accuracy": float}}
    """
    breakdown = {label: {"correct": 0, "total": 0, "accuracy": 0.0} for label in LABELS}
    for r in results:
        expected = r["expected"]
        if expected in breakdown:
            breakdown[expected]["total"] += 1
            if r["correct"]:
                breakdown[expected]["correct"] += 1
    for label in breakdown:
        total = breakdown[label]["total"]
        if total > 0:
            breakdown[label]["accuracy"] = breakdown[label]["correct"] / total
    return breakdown


# Test with base model results from the dry run
if eval_results is not None:
    base_breakdown = per_label_accuracy(eval_results["base_results"])
    ft_breakdown   = per_label_accuracy(eval_results["ft_results"])

    print("Per-label accuracy comparison:")
    print(f"{'Label':12}  {'Base':8}  {'Fine-tuned':12}  {'Delta'}")
    print("-" * 48)
    for label in LABELS:
        b = base_breakdown[label]
        f = ft_breakdown[label]
        delta = f["accuracy"] - b["accuracy"]
        delta_str = f"{delta:+.0%}" if delta != 0 else "  0%"
        print(f"  {label:10}  {b['accuracy']:6.0%}    {f['accuracy']:8.0%}      {delta_str}")

    most_improved = max(LABELS, key=lambda l: ft_breakdown[l]["accuracy"] - base_breakdown[l]["accuracy"])
    print(f"\nMost improved by fine-tuning: '{most_improved}'")
    print("Interpretation: fine-tuning helps most on labels the base model found ambiguous.")
else:
    # Demonstrate with base-only results from DRY_RUN
    print("Exercise 3 — Per-label accuracy (base model, dry run):")
    if "base_only" in dir():
        breakdown = per_label_accuracy(base_only)  # noqa: F821
        print(f"{'Label':12}  {'Correct':8}  {'Total':7}  {'Accuracy'}")
        print("-" * 40)
        for label in LABELS:
            b = breakdown[label]
            print(f"  {label:10}  {b['correct']:6}    {b['total']:5}    {b['accuracy']:.0%}")
        print("\nNote: run with DRY_RUN=False to compare base vs. fine-tuned per label.")
    else:
        print("Run Part 8 cells first to generate base_only results.")

## Workshop Complete

You have built and run a full OpenAI fine-tuning pipeline:

- **JSONL data prep** — balanced synthetic examples, validated format and label distribution
- **Files API upload** — `client.files.create(purpose="fine-tune")` → `file_id`
- **Fine-tuning job** — `client.fine_tuning.jobs.create()` with `n_epochs=3`, dry_run pattern to demo without cost
- **Polling** — loop on `client.fine_tuning.jobs.retrieve()` until `succeeded`
- **Evaluation** — `classify_message()` on both models, exact-match accuracy, per-label breakdown
- **Decision framework** — when fine-tuning is worth the data collection and training investment

**The key insight**: fine-tuning is not always better than prompting — but when your task is narrow, stable, and runs at high volume, baking the task into the model's weights eliminates per-call system prompt overhead and produces more consistent outputs. Start with prompting, build your eval set first, then fine-tune only when the measurement shows a gap worth closing.

---

### Further reading

- [OpenAI Fine-Tuning Guide](https://platform.openai.com/docs/guides/fine-tuning) — official docs with hyperparameter guidance and best practices
- [OpenAI Fine-Tuning Pricing](https://openai.com/pricing) — current training and inference costs per model
- [OpenAI Cookbook: Fine-Tuning](https://cookbook.openai.com/examples/how_to_finetune_chat_models) — detailed notebook with GPT-3.5 examples
- [When to Fine-Tune vs. Prompt](https://platform.openai.com/docs/guides/fine-tuning/when-to-use-fine-tuning) — OpenAI's own decision guidance